# Maze spatial structure: mEC, LEC and PFC through one pipeline

Reads merged tables and shards of one `run_spatial.py --engine maze` run per dataset; it never opens a
data_dic (`RUN_MODE = 'load'`). Write-up: `MAZE_SPATIAL.md`. Galleries of every unit:
`<root>/data/spatial/maze/<tag>/figures/gallery_<recday>.pdf`. The field-shuffle null is the bar for
generalisation claims (2026-10-12).

In [ ]:
import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath('.'))
import maze_spatial_report as r
import spatial_plots as sp
RUN_MODE = 'load'
TAG = '1617859bc6fd'
runs = {d: r.load(d, TAG) for d in r.ROOTS}
{d: (run['stamp'].get('cfg_hash', '')[:10], len(run['recdays']), len(run['units'])) for d, run in runs.items()}

## PFC thesis recipe against the thesis targets (t across recdays, and across mice)

In [ ]:
r.thesis_reproduction(runs['pfc']['population']).round(3)

In [ ]:
r.thesis_planes('pfc', TAG).round(1)

## Decided recipe: mice beyond each null p95, on the generalisation side of chance

In [ ]:
pd.concat([r.claim_counts(d, run) for d, run in runs.items()])

## Per mouse: real value against the field-shuffle null (mean to p95)

In [ ]:
pop = pd.concat([run['population'].assign(dataset=d) for d, run in runs.items()])
nl = pd.concat([run['nulls'].assign(dataset=d) for d, run in runs.items()])
nl = nl[~((nl.dataset == 'mec') & (nl.mouse == 'bp02'))]
fig = sp.plot_population_vs_null(pop, nl, ['rsa9_ED_X', 'dec_node_bridge_ba', 'dec_corner_cardinal_ba',
                                           'dec_same_type_error_share', 'rsa21_contrast'], null_kind='place_shuffled')

## Plane angles (k = 3): across goal-progress bins against the halves and tasks controls

In [ ]:
cols = ['planes_k3_halves_max', 'planes_k3_tasks_max', 'planes_k3_across_max']
pd.concat({d: run['population'][run['population'].recipe == 'decided'].groupby('mouse')[cols].mean()
           for d, run in runs.items()}).round(1)

## Single units per mouse

In [ ]:
def unit_counts(u):
    g = u.groupby('mouse')
    return pd.DataFrame({'units': g.size(), 'stable': g.stable.sum(),
                         **{k: g[f'is_{k}'].sum() for k in ('corner', 'cardinal', 'centre', 'node', 'bridge')}})
pd.concat({d: unit_counts(run['units']) for d, run in runs.items()})